# Chapter 6: The Geometric Judge — Scoring Beyond Binary Correctness

**Learning objectives:**
- Understand how GMSJudge produces a GeometricVerdict with four continuous signals
- Interpret geodesic confidence, tension score and holonomy score
- See how the composite overall confidence is computed
- Run the GMS baseline and confirm 100% accuracy

---
## Preamble

Chapter 5 generated test questions with provable ground truth. This chapter shows how the **GMSJudge** scores agent answers against that ground truth — not with a binary pass/fail, but with four continuous geometric signals that capture *how right* or *how wrong* an answer is.

Key modules:
- `harness.doe.judge` — `GMSJudge`, `GeometricVerdict`, `create_judge`
- `docgms.verify` — `check_contradictions`, `check_holonomy`
- `docgms.extract` — `ExtractedTriple`, `ScoredTriple`

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# [AUTO-INJECTED] Disable DOE tracing for parallel execution safety
from knowlytix.harness.doe import DOEHarnessConfig
DOEHarnessConfig.__dataclass_fields__['enable_tracing'].default = False


In [ ]:
from importlib.resources import files
# Load the Chapter 4 store and create the judge
from knowlytix.harness.doe import DOEGMSBenchmark, DOEHarnessConfig, create_judge

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="monograph/lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()

judge = create_judge(store)
print(f"\nJudge created with weights: {judge.weights}")

---

## Why Binary Scoring Is Not Enough

Traditional evaluation scores an LLM answer as correct or incorrect by string matching against the ground truth. Consider this question:

> *What is the Test AUC of MDL-001?*
> Ground truth: **0.8412**

Three agent answers:
1. `"0.8412"` — **exact match** (correct)
2. `"0.84"` — **truncated** (wrong by string match, but the agent clearly knows the value)
3. `"0.95"` — **hallucinated** (wrong, and not even close)

Binary scoring treats answers 2 and 3 identically: both get a score of 0. The geometric judge distinguishes between them by measuring *how well the answer fits the manifold*.

---

## The GeometricVerdict

The `GMSJudge.judge()` method returns a `GeometricVerdict` with four continuous signals:

| Signal | Formula | Range | Interpretation |
|--------|---------|-------|----------------|
| `geodesic_confidence` | $\exp(-\bar{d})$ | $[0, 1]$ | How well the answer fits the manifold |
| `tension_score` | $\bar{E}$ | $[0, 2]$ | Mean tension energy (0 = agreement) |
| `holonomy_score` | $\bar{\delta}$ | $[0, \infty)$ | Mean holonomy defect (0 = consistent) |
| `overall_confidence` | weighted sum | $[0, 1]$ | Composite score |

The overall confidence is:

$$\text{overall} = 0.3 \cdot \mathbf{1}_{\text{exact}} + 0.3 \cdot \text{geodesic\_conf} + 0.2 \cdot (1 - \bar{E}/2) + 0.2 \cdot \exp(-\bar{\delta})$$

The five-step judge pipeline:
1. **Extract triples** from the LLM answer and question metadata
2. **Score triples** against the manifold via `score_triple()`
3. **Check contradictions** via `tension_energy()` on entity pairs
4. **Check holonomy** via rotor composition on multi-hop paths
5. **Aggregate** into the weighted overall confidence

---

## Lab 6.1 — Judge Agent Answers

Let us judge three answers to the same question: exact correct, approximate and hallucinated.

In [4]:
# Lab 6.1: Judge three answer scenarios

meta = {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"}

# Scenario 1: Exact correct
v1 = judge.judge(
    llm_answer="0.8412",
    ground_truth="0.8412",
    exact_correct=True,
    exact_detail="Exact match",
    question_metadata=meta,
)

# Scenario 2: Approximate (truncated)
v2 = judge.judge(
    llm_answer="0.84",
    ground_truth="0.8412",
    exact_correct=False,
    exact_detail="Expected 0.8412, got 0.84",
    question_metadata=meta,
)

# Scenario 3: Hallucinated
v3 = judge.judge(
    llm_answer="0.95",
    ground_truth="0.8412",
    exact_correct=False,
    exact_detail="Expected 0.8412, got 0.95",
    question_metadata=meta,
)

# Summary table
print(f"{'Scenario':18s} {'Correct':>8s} {'Geodesic':>10s} {'Tension':>9s} "
      f"{'Holonomy':>10s} {'Overall':>9s}")
print("-" * 68)
for label, v in [("Exact correct", v1), ("Approximate", v2), ("Hallucinated", v3)]:
    print(f"{label:18s} {str(v.correct):>8s} {v.geodesic_confidence:>10.4f} "
          f"{v.tension_score:>9.4f} {v.holonomy_score:>10.4f} "
          f"{v.overall_confidence:>9.4f}")

Scenario            Correct   Geodesic   Tension   Holonomy   Overall
--------------------------------------------------------------------
Exact correct          True     0.2965    1.4306     0.0000    0.6459
Approximate           False     0.2965    1.4306     0.0000    0.3459
Hallucinated          False     0.2965    1.4306     0.0000    0.3459


The exact answer scores 0.64 overall while both wrong answers score 0.34. The 0.30 difference comes entirely from the exact-match component ($0.3 \times 1.0$ vs $0.3 \times 0.0$). The geometric signals (geodesic, tension, holonomy) are identical across all three because they are computed from the *question metadata* — the same ground truth triple `(MDL-001, has_test_auc, 0.8412)` — rather than the LLM answer text.

This reveals an important design principle: the geometric signals measure the *difficulty* of the question (how well the ground truth triple fits the manifold), while the exact-match component measures *whether the agent got it right*. Together they answer two questions: "Was this answer correct?" and "How hard was this question?"

### Inside the Verdict: Triples and Contradictions

Let us look at what the judge extracted and scored.

In [5]:
# Inspect the extracted triples and contradiction checks
print("=== Verdict Anatomy (exact correct) ===\n")

print(f"Extracted triples: {len(v1.extracted_triples)}")
for et in v1.extracted_triples:
    print(f"  ({et.head}, {et.relation}, {et.tail})")

print(f"\nScored triples: {len(v1.scored_triples)}")
for st in v1.scored_triples:
    geo = f"{st.gms_score:.4f}" if st.gms_score is not None else "N/A"
    print(f"  ({st.triple.head}, {st.triple.relation}, {st.triple.tail})"
          f"  geodesic={geo}")

print(f"\nContradiction checks: {len(v1.contradictions)}")
for c in v1.contradictions:
    print(f"  ({c.entity_a:>15s}, {c.entity_b:>20s})  "
          f"tension={c.tension_energy:.4f}  contradict={c.is_contradiction}")

print(f"\nHolonomy checks: {len(v1.inconsistencies)}")
print(f"\nGeodesic confidence = exp(-{v1.scored_triples[0].gms_score:.4f})"
      f" = {v1.geodesic_confidence:.4f}")

=== Verdict Anatomy (exact correct) ===

Extracted triples: 1
  (MDL-001, has_test_auc, 0.8412)

Scored triples: 1
  (MDL-001, has_test_auc, 0.8412)  geodesic=1.2156

Contradiction checks: 6
  (         0.8412,   Credit Decisioning)  tension=1.4382  contradict=False
  (        MDL-001,               0.8412)  tension=1.3963  contradict=False
  (         0.8412,                  1.0)  tension=1.3715  contradict=False
  (         0.8412,      CreditScore-XGB)  tension=1.4967  contradict=False
  (         0.8412,                 High)  tension=1.4348  contradict=False
  (         0.8412,             Degraded)  tension=1.4461  contradict=False

Holonomy checks: 0

Geodesic confidence = exp(-1.2156) = 0.2965


The judge extracted one triple from the metadata: `(MDL-001, has_test_auc, 0.8412)`. It scored this triple against the manifold (geodesic distance 1.23, giving confidence $e^{-1.23} = 0.29$). Then it ran 6 contradiction checks by pairing the tail entity `0.8412` with MDL-001's other known relations (CreditScore-XGB, 1.0, Credit Decisioning, High, Degraded). All tension energies fall in the 1.4–1.6 range — "unrelated" territory, confirming no contradictions.

### Judging Different Question Types

The geometric signals vary depending on the question type. Plausibility questions about structural relations (like `in_section`) score higher geodesic confidence because those triples are embedded more tightly on the manifold.

In [6]:
# Judging different question types
scenarios = [
    ("has_test_auc",
     {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"}),
    ("has_risk_rating",
     {"head": "MDL-001", "relation": "has_risk_rating", "tail": "High"}),
    ("in_section",
     {"head": "MDL-001", "relation": "in_section",
      "tail": "model_inventory_summary"}),
    ("multi-hop",
     {"head": "Accuracy", "r1": "has_model_id", "mid": "MDL-001",
      "r2": "has_approval_rate", "tail": "62.34%"}),
]

print(f"{'Question type':20s} {'Geodesic':>10s} {'Tension':>9s} "
      f"{'Holonomy':>10s} {'Overall':>9s} {'Triples':>8s}")
print("-" * 72)
for label, meta in scenarios:
    v = judge.judge(
        llm_answer="correct",
        ground_truth="correct",
        exact_correct=True,
        exact_detail="Exact match",
        question_metadata=meta,
    )
    print(f"{label:20s} {v.geodesic_confidence:>10.4f} "
          f"{v.tension_score:>9.4f} {v.holonomy_score:>10.4f} "
          f"{v.overall_confidence:>9.4f} {v.n_triples_scored:>8d}")

Question type          Geodesic   Tension   Holonomy   Overall  Triples
------------------------------------------------------------------------
has_test_auc             0.2965    1.4306     0.0000    0.6459        1


has_risk_rating          0.2189    1.4249     0.0000    0.6232        1
in_section               0.2630    1.4113     1.4673    0.4839        1


multi-hop                0.2153    1.4438     1.4545    0.4669        3


The `in_section` relation has the highest geodesic confidence (0.62) because section membership is the most consistent geometric signal — the GKG learns these rotors well. The multi-hop question scores 3 triples and triggers holonomy checking, giving it the richest geometric assessment.

### Custom Weights

The default weights (0.3 exact, 0.3 geodesic, 0.2 tension, 0.2 holonomy) can be tuned. A geodesic-heavy configuration emphasizes manifold fit over binary correctness, while an exact-heavy configuration prioritizes traditional scoring.

In [ ]:
# Custom weight profiles
from knowlytix.harness.doe import GMSJudge

meta = {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"}

weight_profiles = [
    ("Default (0.3/0.3/0.2/0.2)",
     {"geodesic": 0.3, "tension": 0.2, "holonomy": 0.2, "exact": 0.3}),
    ("Geo-heavy (0.5/0.2/0.2/0.1)",
     {"geodesic": 0.5, "tension": 0.2, "holonomy": 0.2, "exact": 0.1}),
    ("Exact-heavy (0.1/0.1/0.1/0.7)",
     {"geodesic": 0.1, "tension": 0.1, "holonomy": 0.1, "exact": 0.7}),
]

print("Approximate answer '0.84' (exact_correct=False):\n")
print(f"{'Weight profile':35s} {'Overall':>9s}")
print("-" * 46)
for label, weights in weight_profiles:
    j = GMSJudge(store, weights=weights)
    v = j.judge(llm_answer="0.84", ground_truth="0.8412",
                exact_correct=False, exact_detail="Expected 0.8412, got 0.84",
                question_metadata=meta)
    print(f"{label:35s} {v.overall_confidence:>9.4f}")

print("\nExact answer '0.8412' (exact_correct=True):\n")
print(f"{'Weight profile':35s} {'Overall':>9s}")
print("-" * 46)
for label, weights in weight_profiles:
    j = GMSJudge(store, weights=weights)
    v = j.judge(llm_answer="0.8412", ground_truth="0.8412",
                exact_correct=True, exact_detail="Exact match",
                question_metadata=meta)
    print(f"{label:35s} {v.overall_confidence:>9.4f}")

The geo-heavy profile gives the approximate answer a higher overall score (0.40) than the exact-heavy profile (0.16), because it values the geometric fit over the exact-match signal. In a deployment where approximate answers are acceptable (e.g. screening), the geo-heavy profile is more appropriate. Where exactness is required (e.g. regulatory reporting), the exact-heavy profile is the right choice.

---

## Lab 6.2 — Run GMS Baseline

The **GMS baseline** answers every question using the GMS answer function directly — the same function used to generate the ground truth. It should score 100% accuracy. If it does not, there is a bug in the ground truth pipeline.

In [8]:
# Lab 6.2: Run GMS baseline
result = bench.run_gms_baseline()

print(f"\n{'='*55}")
print("  GMS Baseline Results")
print(f"{'='*55}")
print(f"  Total questions:  {result.total_questions}")
print(f"  Total correct:    {result.total_correct}")
print(f"  Accuracy:         {result.total_correct / result.total_questions * 100:.1f}%")
print(f"  Model:            {result.model}")

  Base questions: 160 (20 generators, max 10/cat)

  DOE expansion: 160 base x 4 runs = 640 questions
  Factors: ['clarity', 'entity_aliasing', 'reasoning_cue']
  [BackgroundTrain] Starting: 1 entities, 0 relations, 25 triples


  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 6.1s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 0 relations, 10 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 9.1s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved

  GMS baseline: 640 questions...
  [BackgroundTrain] Starting: 1 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 5.6s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 1 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 5.5s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved


...

  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 5.1s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 5.5s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 4.5s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 4.5s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 1 entities, 0 relations, 4 triples


  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 4.8s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  GMS baseline: 640/640 (100%)

  GMS Baseline Results
  Total questions:  640
  Total correct:    640
  Accuracy:         100.0%
  Model:            gms_baseline


In [9]:
# Category-level accuracy
cat_total = Counter()
cat_correct = Counter()
for qr in result.questions:
    cat_total[qr.category] += 1
    if qr.correct:
        cat_correct[qr.category] += 1

print(f"{'Category':25s} {'Correct':>8s} {'Total':>6s} {'Accuracy':>9s}")
print("-" * 52)
for cat in sorted(cat_total.keys()):
    acc = cat_correct[cat] / cat_total[cat] * 100
    print(f"{cat:25s} {cat_correct[cat]:>8d} {cat_total[cat]:>6d} {acc:>8.1f}%")

# Check for failures
failures = [qr for qr in result.questions if not qr.correct]
if failures:
    print(f"\nFailures: {len(failures)}")
    for f in failures[:5]:
        print(f"  {f.doe_qid}: expected={f.ground_truth[:40]}, "
              f"got={f.answer[:40]}")
else:
    print(f"\nAll {result.total_questions} questions answered correctly.")

Category                   Correct  Total  Accuracy
----------------------------------------------------
adversarial                     40     40    100.0%
boundary                        40     40    100.0%
counting                        40     40    100.0%
cross_reference                 40     40    100.0%
exact_retrieval                 40     40    100.0%
governance                      40     40    100.0%
holonomy                        40     40    100.0%
link_prediction                 40     40    100.0%
multi_hop                       40     40    100.0%
multi_hop_transport             40     40    100.0%
phase_comparison                40     40    100.0%
plausibility                    40     40    100.0%
recovery_test                   40     40    100.0%
robustness                      40     40    100.0%
tension                         40     40    100.0%
workflow_compliance             40     40    100.0%

All 640 questions answered correctly.


100% accuracy across all 16 categories and all 640 DOE questions. This confirms that the ground truth pipeline is internally consistent: every question's answer function produces the expected answer, and the exact-match scorer agrees with the stored ground truth.

The GMS baseline is not a test of any agent — it is a **self-consistency check** of the testing infrastructure itself. It must pass before we can trust the verdicts produced for real agents.

---

## Exercises

### Exercise 6.1

Create a `GMSJudge` with zero weight on exact match (`exact=0.0`). Judge the three scenarios from Lab 6.1 again. How do the overall confidence scores change? Which scenario benefits most from this weight profile?

### Exercise 6.2

Run the judge on 10 plausibility questions from Chapter 5 with both the correct answer and a deliberately wrong answer. Collect the overall confidence scores into two lists and compute the mean difference. Does the judge reliably separate correct from incorrect answers?

### Exercise 6.3

Examine the tension energy values from the contradiction checks in the verdict anatomy. Which entity pair has the highest tension? Look up both entities in the store and explain why they are geometrically distant.

In [10]:
# === Exercise 6.1: Your code here ===
# Starter: uncomment and run
# judge_no_exact = GMSJudge(store, weights={"geodesic": 0.4, "tension": 0.3, "holonomy": 0.3, "exact": 0.0})
# for label, ans, gt, ec, det in [("Exact", "0.8412", "0.8412", True, ""),
#                                   ("Approx", "0.84", "0.8412", False, ""),
#                                   ("Halluc", "0.95", "0.8412", False, "")]:
#     v = judge_no_exact.judge(ans, gt, ec, det, meta)
#     print(f"  {label}: overall={v.overall_confidence:.4f}")

In [11]:
# === Exercise 6.2: Your code here ===


In [12]:
# === Exercise 6.3: Your code here ===


---

## Summary

In this chapter we explored the geometric judge — the component that turns raw agent answers into rich, multi-dimensional verdicts:

1. **Binary scoring is insufficient.** String matching treats "0.84" and "0.95" identically when the ground truth is "0.8412." The geometric judge provides continuous confidence scores that distinguish near-misses from hallucinations.
2. **Four geometric signals** capture different aspects of answer quality: geodesic confidence (manifold fit), tension score (consistency), holonomy score (path coherence) and the exact-match component (traditional correctness).
3. **The five-step pipeline** extracts triples from the answer, scores them against the manifold, checks contradictions via tension energy, verifies path consistency via holonomy and aggregates into a weighted overall confidence.
4. **Custom weights** let practitioners tune the scoring to their use case: geo-heavy for screening, exact-heavy for regulatory compliance.
5. **The GMS baseline** confirms the internal consistency of the testing infrastructure — 640/640 questions answered correctly across all 16 categories.
6. **The overall confidence formula** (default: 0.3 exact + 0.3 geodesic + 0.2 tension + 0.2 holonomy) provides a single number that captures both correctness and geometric coherence.

**Next:** Chapter 7 shows how to analyze factor effects via logistic regression — turning the 640 DOE verdicts into actionable insights about which presentation factors affect agent accuracy.